# Abdomen CT — Swin Transformer Multi-Label Sınıflandırma
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Görev:** 6 acil karın patolojisini kesit bazında multi-label sınıflandırma  
**Model:** `swin_base_patch4_window12_384` (ImageNet-22k pretrained, 87M parametre)  
**Girdi:** DICOM → 3-kanallı HU pencereli NPZ (soft_tissue · liver · calcified)  

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle

---

| Adım | Süre (T4) |
|------|----------|
| NPZ Export (DICOM → 3ch float16) | ~40–60 dk |
| Eğitim (50 epoch, batch=16) | ~3–4 saat |
| Değerlendirme | ~5 dk |

> **Session kesilirse:** `SWIN_CONTINUE = True` yapıp NPZ adımını atlayın, eğitime devam edin.

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

print(f"Ortam : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Yerel'}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU bulunamadı!\n"
        "Kaggle: Settings → Accelerator → GPU\n"
        "Colab : Runtime → Change runtime type → GPU"
    )

print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")

---
## 1. Ortam Kurulumu (Colab için)

In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")
else:
    print("Kaggle ortamı — kurulum atlandı")

---
## 2. Kütüphane Kurulumu

In [ ]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'timm', 'pydicom', 'opencv-python-headless',
     'pandas', 'numpy', 'tqdm', 'scikit-learn'],
    check=True
)
import importlib; importlib.invalidate_caches()

import timm
print(f"timm    : {timm.__version__}")
import torch
print(f"PyTorch : {torch.__version__}")
import pydicom
print(f"pydicom : {pydicom.__version__}")

---
## 3. Konfigürasyon

In [ ]:
import os, sys, json, shutil, time, math, random, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
FOLD = 0

@dataclass
class SwinConfig:
    backbone: str = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k'
    num_classes: int = 6
    input_size: int = 384      # window12 ile tam uyumlu (384/12=32)
    batch_size: int = 16       # T4/P100 için
    epochs: int = 50
    lr: float = 2e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 3
    focal_gamma: float = 2.0
    mixup_alpha: float = 0.0   # CT'de kapalı
    accum_steps: int = 2       # eff batch = 32
    threshold: float = 0.5     # sigmoid eşiği

SWIN_CFG = SwinConfig()
# ──────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis',
    'kidney_ureter_stone',
    'acute_pancreatitis',
    'aortic_aneurysm_dissection',
    'acute_appendicitis',
    'acute_diverticulitis',
]

@dataclass(frozen=True)
class Window:
    name: str
    level: float
    width: float

DEFAULT_WINDOWS: Tuple[Window, ...] = (
    Window('soft_tissue', level=40,  width=400),
    Window('liver',       level=30,  width=150),
    Window('calcified',   level=450, width=1500),
)

# Ortama göre yollar
if IS_KAGGLE:
    DATA_DIR    = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR    = Path('/kaggle/working')
    DRIVE_BASE  = None
elif IS_COLAB:
    DATA_DIR    = Path('/content/kaggle_data')
    WORK_DIR    = Path('/content')
    DRIVE_BASE  = Path('/content/drive/MyDrive/Abdomen')
    if DRIVE_BASE:
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)

EGITIM_DIR  = DATA_DIR / 'Egitim Verisi'
YARISMA_DIR = DATA_DIR / 'Test Verisi'
SPLIT_DIR   = DATA_DIR / 'outputs' / 'splits'
NPZ_DIR     = WORK_DIR / 'cls_data'
CKPT_DIR    = WORK_DIR / 'checkpoints' / f'swin_fold{FOLD}'
LOG_DIR     = WORK_DIR / 'logs'
OUTPUT_DIR  = WORK_DIR / 'output'

for d in [NPZ_DIR, CKPT_DIR, LOG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Konfigürasyon:')
print(f'  Ortam     : {"Kaggle" if IS_KAGGLE else "Colab"}')
print(f'  DATA_DIR  : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'  SPLIT_DIR : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
print(f'  NPZ_DIR   : {NPZ_DIR}')
print(f'  Backbone  : {SWIN_CFG.backbone}')
print(f'  Epochs    : {SWIN_CFG.epochs}  batch={SWIN_CFG.batch_size}  accum={SWIN_CFG.accum_steps}')

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Dataset bulunamadı: {DATA_DIR}\n'
        f'Kaggle sağ panelden Datasets → "{KAGGLE_DATASET_SLUG}" ekleyin.'
    )

---
## 4. Veri İndirme (Yalnızca Colab)

In [ ]:
if IS_KAGGLE:
    print("Kaggle: Dataset zaten mount edilmiş")
else:
    if DATA_DIR.exists() and EGITIM_DIR.exists():
        print(f"Veri zaten mevcut: {DATA_DIR}")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}  (~30-90 dk)")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-2000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({(time.time()-t0)/60:.0f} dk)")

assert EGITIM_DIR.exists(), f'Egitim Verisi bulunamadı: {EGITIM_DIR}'
assert (SPLIT_DIR / 'manifest.csv').exists(), f'manifest.csv bulunamadı: {SPLIT_DIR}'
print(f'Egitim Verisi vakaları : {len(list(EGITIM_DIR.iterdir()))}')
print(f'manifest.csv satırı   : {len(pd.read_csv(SPLIT_DIR/"manifest.csv")):,}')

---
## 5. DICOM → NPZ Export

Manifest'teki her kesiti 3-kanallı float16 NPZ olarak `/kaggle/working/cls_data/` altına kaydeder.  
**Süre:** ~40–60 dakika (39K kesit)  
Session yeniden başlarsa tekrar çalıştırılması gerekir.

> `SKIP_EXPORT = True` yaparak atlayabilirsiniz (NPZ zaten mevcutsa).

In [ ]:
import warnings
import pydicom
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed

SKIP_EXPORT = False   # True → mevcut NPZ sayısını kontrol et, yeterli ise atla

# ── HU yardımcıları (satır içi) ──────────────────────────────────────────
def _read_dicom(path):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=UserWarning)
        return pydicom.dcmread(str(path), force=True)

def _dicom_to_hu(ds):
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
    hu = arr * slope + intercept
    hu = np.clip(hu, -1024, 3071)
    return hu

def _window_hu(hu, win):
    low = win.level - win.width / 2
    high = win.level + win.width / 2
    return np.clip((hu - low) / max(high - low, 1e-6), 0., 1.).astype(np.float32)

def _hu_to_three_channel(hu, windows=DEFAULT_WINDOWS):
    return np.stack([_window_hu(hu, w) for w in windows], axis=-1)  # HxWx3

def _export_one(row, out_dir):
    """Tek satırı NPZ olarak yazar. (case, image_id) → dosya adı."""
    dpath = Path(row['dicom_path'])
    # Kaggle'da yol farklı olabilir — DATA_DIR ile yeniden oluştur
    # manifest'teki yol yerel makineye aittir; vaka + image_id'den yeniden türet
    case_raw = str(row['case']).split('_', 1)[1]    # 'T_20001' → '20001'
    source = str(row['source'])
    base = EGITIM_DIR if source == 'train' else YARISMA_DIR
    dpath_actual = base / case_raw / f"{row['image_id']}.dcm"
    if not dpath_actual.exists():
        return None, f"yok: {dpath_actual}"
    out = Path(out_dir) / f"{row['case']}_{row['image_id']}.npz"
    if out.exists():
        return str(out), None   # zaten var
    try:
        ds = _read_dicom(dpath_actual)
        hu = _dicom_to_hu(ds)
        img = _hu_to_three_channel(hu)
        labels = np.zeros(len(SUPER_CLASSES), dtype=np.uint8)
        sl = str(row.get('super_labels', ''))
        if sl and sl != 'nan':
            for s in sl.split(';'):
                if s.strip():
                    labels[int(s.strip())] = 1
        np.savez_compressed(
            out,
            image=img.astype(np.float16),
            labels=labels,
            case=str(row['case']),
            image_id=int(row['image_id']),
            source=source,
        )
        return str(out), None
    except Exception as e:
        return None, f"hata: {e}"

# ── Export ───────────────────────────────────────────────────────────────
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
existing = list(NPZ_DIR.glob('*.npz'))
print(f'Manifest  : {len(manifest):,} kesit')
print(f'Mevcut NPZ: {len(existing):,}')

if SKIP_EXPORT and len(existing) >= len(manifest) * 0.95:
    print('NPZ export atlandı (yeterli dosya mevcut)')
else:
    print(f'NPZ export başlıyor → {NPZ_DIR}')
    t0 = time.time()
    rows = [r for _, r in manifest.iterrows()]
    done = err = 0
    pbar = tqdm(total=len(rows), desc='DICOM→NPZ')
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = {ex.submit(_export_one, r, NPZ_DIR): r for r in rows}
        for f in as_completed(futs):
            out, e = f.result()
            if e:
                err += 1
            else:
                done += 1
            pbar.update(1)
    pbar.close()
    elapsed = time.time() - t0
    print(f'Tamamlandı: {done:,} NPZ  ({err} hata)  {elapsed/60:.1f} dk')

npz_count = len(list(NPZ_DIR.glob('*.npz')))
print(f'Toplam NPZ: {npz_count:,}')
if npz_count < 1000:
    raise RuntimeError(f'NPZ sayısı çok düşük ({npz_count}). DICOM klasörünü kontrol edin.')

---
## 6. Yardımcı Fonksiyonlar (Dataset · Loss · Model · Metrics)

In [ ]:
import cv2
from sklearn.metrics import average_precision_score, f1_score

# ── Split yükleme ─────────────────────────────────────────────────────────
def load_fold(fold, split):
    p = SPLIT_DIR / f'fold{fold}_{split}.csv'
    return pd.read_csv(p)['Case Number'].astype(str).tolist()

def load_holdout():
    return pd.read_csv(SPLIT_DIR / 'holdout.csv')['Case Number'].astype(str).tolist()

def count_positives(cases):
    mani = pd.read_csv(SPLIT_DIR / 'manifest.csv')
    mani = mani[mani['case'].astype(str).isin(set(str(c) for c in cases))]
    counts = [0] * len(SUPER_CLASSES)
    for sl in mani['super_labels'].fillna(''):
        for s in str(sl).split(';'):
            if s.strip(): counts[int(s.strip())] += 1
    return counts

# ── Dataset ───────────────────────────────────────────────────────────────
class SliceMultiLabelDataset(Dataset):
    """NPZ → 3ch float32 tensor, multi-label."""

    def __init__(self, case_ids, data_dir=NPZ_DIR, input_size=384):
        self.data_dir = Path(data_dir)
        self.input_size = input_size
        case_set = set(str(c) for c in case_ids)
        self.files: List[Path] = sorted(
            p for p in self.data_dir.glob('*.npz')
            if p.stem.rsplit('_', 1)[0] in case_set
        )
        if not self.files:
            raise RuntimeError(f'{self.data_dir} içinde eşleşen NPZ bulunamadı.')

        # Manifest'ten label lookup (NPZ etiketi yerine)
        self._label_lookup: Dict[tuple, np.ndarray] = {}
        mpath = SPLIT_DIR / 'manifest.csv'
        if mpath.exists():
            mdf = pd.read_csv(mpath)
            mdf['case'] = mdf['case'].astype(str)
            mdf = mdf[mdf['case'].isin(case_set)]
            for _, row in mdf.iterrows():
                vec = np.zeros(len(SUPER_CLASSES), dtype=np.float32)
                sl = str(row.get('super_labels', ''))
                if sl and sl != 'nan':
                    for s in sl.split(';'):
                        if s.strip(): vec[int(s.strip())] = 1.0
                self._label_lookup[(str(row['case']), int(row['image_id']))] = vec

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        with np.load(path, allow_pickle=False) as npz:
            img    = npz['image'].astype(np.float32)      # H,W,3
            labels = npz['labels'].astype(np.float32)
            case   = str(npz['case'])
            image_id = int(npz['image_id'])

        key = (case, image_id)
        if key in self._label_lookup:
            labels = self._label_lookup[key]

        img_chw = np.transpose(img, (2, 0, 1))   # HxWx3 → 3xHxW
        if img_chw.shape[-1] != self.input_size:
            out = np.zeros((3, self.input_size, self.input_size), dtype=np.float32)
            for c in range(3):
                out[c] = cv2.resize(img_chw[c],
                                    (self.input_size, self.input_size),
                                    interpolation=cv2.INTER_LINEAR)
            img_chw = out

        return {
            'image' : torch.from_numpy(img_chw).float(),
            'labels': torch.from_numpy(labels).float(),
        }

# ── Loss ──────────────────────────────────────────────────────────────────
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            if not isinstance(alpha, torch.Tensor):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        focal = (1 - pt) ** self.gamma * bce
        if self.alpha is not None:
            a = self.alpha.to(logits.device)
            w = targets * a + (1 - targets) * (1 - a)
            focal = w * focal
        return focal.mean()

def class_balanced_alpha(pos_counts, beta=0.9999):
    n = torch.tensor(pos_counts, dtype=torch.float64)
    eff = 1.0 - beta ** n
    w = (1.0 - beta) / torch.clamp(eff, min=1e-8)
    w = w / w.sum() * len(pos_counts)
    return torch.sigmoid(torch.log(w)).float()

# ── Model ─────────────────────────────────────────────────────────────────
import timm

def build_model(cfg):
    return timm.create_model(
        cfg.backbone, pretrained=True,
        num_classes=cfg.num_classes, in_chans=3,
    )

# ── Metrikler ─────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_prob, all_true = [], []
    for batch in tqdm(loader, desc='  val', leave=False):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits = model(batch['image'].to(device))
        all_prob.append(logits.float().sigmoid().cpu().numpy())
        all_true.append(batch['labels'].numpy())

    y_prob = np.concatenate(all_prob)
    y_true = np.concatenate(all_true)
    y_pred = (y_prob >= SWIN_CFG.threshold).astype(np.int32)

    metrics = {}
    for i, cls in enumerate(SUPER_CLASSES):
        if y_true[:, i].sum() == 0:
            metrics[f'AP/{cls}'] = float('nan')
            metrics[f'F1/{cls}'] = float('nan')
            continue
        metrics[f'AP/{cls}'] = float(average_precision_score(y_true[:, i], y_prob[:, i]))
        metrics[f'F1/{cls}'] = float(f1_score(y_true[:, i], y_pred[:, i], zero_division=0))

    metrics['mAP']     = float(np.nanmean([metrics[f'AP/{c}'] for c in SUPER_CLASSES]))
    metrics['macroF1'] = float(np.nanmean([metrics[f'F1/{c}'] for c in SUPER_CLASSES]))
    return metrics

def _fmt(s):
    s = int(s); h, r = divmod(s, 3600); m, sc = divmod(r, 60)
    return f'{h}s{m:02d}d{sc:02d}s' if h else f'{m}d{sc:02d}s'

print('Yardımcılar yüklendi ✓')

# Hızlı doğrulama
_train_cases = load_fold(FOLD, 'train')
_val_cases   = load_fold(FOLD, 'val')
print(f'Fold {FOLD}: {len(_train_cases)} train / {len(_val_cases)} val vaka')

---
## 7. Eğitim

Session kesilirse: `SWIN_CONTINUE = True` yapıp tekrar çalıştırın.

In [ ]:
SWIN_CONTINUE = False  # True → son checkpoint'ten devam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_work  = min(8, os.cpu_count() or 4)
pin_mem = device.type == 'cuda'

print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Device : {device}  workers={n_work}')

train_ds = SliceMultiLabelDataset(_train_cases, input_size=SWIN_CFG.input_size)
val_ds   = SliceMultiLabelDataset(_val_cases,   input_size=SWIN_CFG.input_size)
print(f'Train  : {len(train_ds):,} kesit')
print(f'Val    : {len(val_ds):,} kesit')

train_loader = DataLoader(
    train_ds, batch_size=SWIN_CFG.batch_size, shuffle=True,
    num_workers=n_work, pin_memory=pin_mem, drop_last=True,
    persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=SWIN_CFG.batch_size, shuffle=False,
    num_workers=n_work, pin_memory=pin_mem,
    persistent_workers=True, prefetch_factor=4,
)

# Class-balanced loss
pos_counts = count_positives(_train_cases)
alpha      = class_balanced_alpha(pos_counts)
criterion  = FocalBCE(gamma=SWIN_CFG.focal_gamma, alpha=alpha)
print(f'pos_counts : {pos_counts}')
print(f'alpha      : {[f"{a:.3f}" for a in alpha.tolist()]}')

# Model
model = build_model(SWIN_CFG).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model  : {SWIN_CFG.backbone}  ({n_params:.0f}M param)')

optim = AdamW(model.parameters(), lr=SWIN_CFG.lr,
               weight_decay=SWIN_CFG.weight_decay)

warmup_ep  = max(1, SWIN_CFG.warmup_epochs)
cosine_ep  = max(1, SWIN_CFG.epochs - warmup_ep)
sched = SequentialLR(
    optim,
    schedulers=[
        LinearLR(optim, start_factor=1e-3, end_factor=1.0, total_iters=warmup_ep),
        CosineAnnealingLR(optim, T_max=cosine_ep, eta_min=SWIN_CFG.lr * 1e-3),
    ],
    milestones=[warmup_ep],
)

scaler = torch.amp.GradScaler('cuda')
accum  = max(1, SWIN_CFG.accum_steps)

start_epoch = 0
best = {'mAP': -1., 'macroF1': 0., 'epoch': -1}

# Resume
_ckpt = CKPT_DIR / 'last.pth'
if SWIN_CONTINUE and _ckpt.exists():
    ck = torch.load(str(_ckpt), map_location=device)
    model.load_state_dict(ck['model'])
    optim.load_state_dict(ck['optim'])
    sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    best        = ck['best']
    print(f'Checkpoint yüklendi → epoch={start_epoch}  mAP={best["mAP"]:.4f}')

steps_per_ep = len(train_loader)
total_steps  = SWIN_CFG.epochs * steps_per_ep
log_rows = []

print(f'\n{"="*60}')
print(f'  EĞİTİM  Fold {FOLD}  │  {SWIN_CFG.epochs} epoch  (başlangıç={start_epoch})')
print(f'  Backbone  : {SWIN_CFG.backbone}')
print(f'  Batch eff.: {SWIN_CFG.batch_size * accum}')
print(f'  LR        : {SWIN_CFG.lr} → {SWIN_CFG.lr*1e-3:.2e}')
print(f'{"="*60}\n')

t_start = time.time()

for epoch in range(start_epoch, SWIN_CFG.epochs):
    model.train()
    t0 = time.time()
    running = 0.
    optim.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader,
                desc=f'[F{FOLD}] Ep {epoch+1:02d}/{SWIN_CFG.epochs}',
                unit='bat', leave=True)

    for step, batch in enumerate(pbar):
        imgs    = batch['image'].to(device, non_blocking=True)
        targets = batch['labels'].to(device, non_blocking=True)

        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits = model(imgs)
            loss   = criterion(logits, targets) / accum

        scaler.scale(loss).backward()

        if (step + 1) % accum == 0 or (step + 1) == steps_per_ep:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)

        running += loss.item() * accum
        done = epoch * steps_per_ep + step + 1
        eta  = (time.time() - t_start) / done * (total_steps - done) if done > 0 else 0
        pbar.set_postfix({
            'loss': f'{running/(step+1):.4f}',
            'lr'  : f'{optim.param_groups[0]["lr"]:.2e}',
            'ETA' : _fmt(eta),
        }, refresh=False)

    pbar.close()
    sched.step()
    train_loss = running / steps_per_ep

    # Validation
    metrics = evaluate(model, val_loader, device)
    ep_sec = time.time() - t0
    metrics.update({'train_loss': train_loss, 'epoch': epoch, 'epoch_sec': ep_sec})
    log_rows.append(metrics)

    print(f'\n── Fold {FOLD}  Epoch {epoch:02d}  │  mAP={metrics["mAP"]:.4f}  macroF1={metrics["macroF1"]:.4f}')
    for cls in SUPER_CLASSES:
        ap = metrics.get(f'AP/{cls}', float('nan'))
        f1v = metrics.get(f'F1/{cls}', float('nan'))
        print(f'  {cls:<35} AP={ap:.4f}  F1={f1v:.4f}')
    print(f'  ⏱ {ep_sec:.0f}s  Kalan: {_fmt((time.time()-t_start)/(epoch-start_epoch+1)*(SWIN_CFG.epochs-epoch-1))}')

    # Best model save
    if metrics['mAP'] > best['mAP']:
        best = {'mAP': metrics['mAP'], 'macroF1': metrics['macroF1'], 'epoch': epoch}
        torch.save({'model': model.state_dict(), 'cfg': SWIN_CFG.__dict__,
                    'epoch': epoch, 'metrics': metrics}, CKPT_DIR / 'best.pth')
        print(f'  ✅ En iyi güncellendi → mAP={best["mAP"]:.4f}')

    # Last checkpoint (resume için)
    torch.save({
        'model': model.state_dict(), 'optim': optim.state_dict(),
        'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
        'epoch': epoch, 'best': best,
    }, CKPT_DIR / 'last.pth')

    pd.DataFrame(log_rows).to_csv(LOG_DIR / f'swin_fold{FOLD}.csv', index=False)

print(f'\n{"="*60}')
print(f'  EĞİTİM TAMAMLANDI  ({_fmt(time.time()-t_start)})')
print(f'  En iyi  → epoch={best["epoch"]:02d}  mAP={best["mAP"]:.4f}  macroF1={best["macroF1"]:.4f}')
print(f'{"="*60}')

BEST_CKPT = CKPT_DIR / 'best.pth'
print(f'Checkpoint: {BEST_CKPT}')

if IS_COLAB and DRIVE_BASE:
    _yd = DRIVE_BASE / 'swin_weights'
    _yd.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(BEST_CKPT), str(_yd / f'fold{FOLD}_best.pth'))
    print(f"Drive'a yedeklendi: {_yd}")

---
## 8. Validation Değerlendirmesi

In [ ]:
# Checkpoint manuel yüklemek için:
# BEST_CKPT = Path('/kaggle/working/checkpoints/swin_fold0/best.pth')
assert 'BEST_CKPT' in dir() and Path(BEST_CKPT).exists(), f'Checkpoint yok: {BEST_CKPT}'

ck = torch.load(str(BEST_CKPT), map_location=device)
_eval_model = build_model(SWIN_CFG).to(device)
_eval_model.load_state_dict(ck['model'])
print(f"Checkpoint: epoch={ck['epoch']}  mAP={ck['metrics']['mAP']:.4f}")

val_metrics = evaluate(_eval_model, val_loader, device)

print(f'\n=== Validation Fold {FOLD} ===')
print(f'mAP    : {val_metrics["mAP"]:.4f}')
print(f'macroF1: {val_metrics["macroF1"]:.4f}')
print()
print(f'{"Sınıf":<35} {"AP":>7}  {"F1":>7}')
print('─' * 52)
for cls in SUPER_CLASSES:
    ap  = val_metrics.get(f'AP/{cls}', float('nan'))
    f1v = val_metrics.get(f'F1/{cls}', float('nan'))
    print(f'{cls:<35} {ap:>7.4f}  {f1v:>7.4f}')

---
## 9. Eğitim Grafiği

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

_log_csv = LOG_DIR / f'swin_fold{FOLD}.csv'
if not _log_csv.exists():
    print('Log CSV bulunamadı, eğitim adımını çalıştırın.')
else:
    _df = pd.read_csv(_log_csv)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(_df['epoch'], _df['train_loss'], label='train loss')
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)

    axes[1].plot(_df['epoch'], _df['mAP'],     label='mAP')
    axes[1].plot(_df['epoch'], _df['macroF1'], label='macroF1')
    axes[1].set_title('Val mAP / macroF1'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True)

    ap_cols = [c for c in _df.columns if c.startswith('AP/')]
    for col in ap_cols:
        axes[2].plot(_df['epoch'], _df[col], label=col.replace('AP/', ''))
    axes[2].set_title('Per-Class AP'); axes[2].set_xlabel('Epoch')
    axes[2].legend(fontsize=7); axes[2].grid(True)

    plt.suptitle(f'Swin Transformer — Fold {FOLD}', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'swin_fold{FOLD}_curves.png', dpi=120)
    plt.show()

---
## 10. Sonuçları Kaydet

In [ ]:
summary = {
    'fold'      : FOLD,
    'backbone'  : SWIN_CFG.backbone,
    'best_epoch': best['epoch'],
    'mAP'       : best['mAP'],
    'macroF1'   : best['macroF1'],
    'val_metrics': {
        k: round(v, 4) if isinstance(v, float) and not math.isnan(v) else v
        for k, v in val_metrics.items()
    },
    'config': SWIN_CFG.__dict__,
}
(OUTPUT_DIR / f'swin_fold{FOLD}_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False)
)

print(f'Çıktılar: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

print()
print('=' * 50)
print(f'  Swin Transformer Fold {FOLD} — Özet')
print('=' * 50)
print(f'  Backbone   : {SWIN_CFG.backbone}')
print(f'  Best epoch : {best["epoch"]}')
print(f'  mAP        : {best["mAP"]:.4f}')
print(f'  macroF1    : {best["macroF1"]:.4f}')
print('=' * 50)

if IS_COLAB and DRIVE_BASE:
    _dst = DRIVE_BASE / 'output'
    _dst.mkdir(parents=True, exist_ok=True)
    for f in OUTPUT_DIR.glob('swin_*'):
        shutil.copy2(str(f), str(_dst / f.name))
    print(f"Drive'a kopyalandı: {_dst}")

if IS_KAGGLE:
    print('\nKaggle: Save Version → sağ üst → Save & Run All')